# CNN Preprocessing & Data Augmentation - Complete Guide

Image data ကို CNN model ထဲ ထည့်မပို့ခင် preprocessing stage မှာ လုပ်ရမယ့် အဆင့်တွေ အကုန်လုံးကို visualize လုပ်ပြမယ်။

## Part A: Core Transforms (Training Pipeline)
1. `RandomCrop(32, padding=4)` — padding ထည့်ပြီး random crop ဖြတ်
2. `RandomHorizontalFlip()` — 50% chance နဲ့ ဘယ်/ညာ flip
3. `ToTensor()` — PIL Image → Tensor (pixel [0,255] → [0,1])
4. `Normalize(mean, std)` — channel-wise standardization (mean≈0, std≈1)

## Part B: Additional Preprocessing Techniques
5. `Resize()` & `CenterCrop()` — image size ညီအောင် ပြောင်း
6. `ColorJitter()` — brightness, contrast, saturation, hue ပြောင်း
7. `RandomRotation()` — image ကို rotate ဘယ်ညာ လှည့်
8. `RandomErasing()` — image ရဲ့ part ကို erase/mask လုပ်
9. `GaussianBlur()` — blur effect
10. `Grayscale()` — color → grayscale ပြောင်း
11. **Custom Mean/Std Calculation** — ကိုယ့် dataset အတွက် mean, std တွက်ခြင်း

In [ ]:
# --- Imports ---
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from PIL import Image

# --- CIFAR-10 ကို transform မသုံးဘဲ raw PIL Image အနေနဲ့ load ---
# transform=None ဆိုရင် PIL Image (32x32) အတိုင်း ရမယ်
raw_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# ပထမ image ကို ယူမယ်
original_img, label = raw_dataset[0]  # PIL Image, int
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Type: {type(original_img)}")
print(f"Size: {original_img.size}")          # (width, height)
print(f"Label: {class_names[label]}")
print(f"Pixel range: [0, 255] (PIL Image)")

## Step 1: Original Image (No Transform)

Raw PIL Image — ဘာ transform မှ မလုပ်ရသေးတဲ့ မူရင်း image

In [ ]:
# --- Original PIL Image ကို ပြမယ် ---
plt.figure(figsize=(3, 3))
plt.imshow(original_img)
plt.title(f"Original PIL Image\n{class_names[label]} | size={original_img.size}")
plt.axis('off')
plt.show()

# PIL Image ရဲ့ pixel values ကို ကြည့်မယ်
img_array = np.array(original_img)
print(f"Shape (H, W, C): {img_array.shape}")   # (32, 32, 3) — numpy array
print(f"Pixel range: [{img_array.min()}, {img_array.max()}]")  # [0, 255]
print(f"Dtype: {img_array.dtype}")               # uint8

## Step 2: RandomCrop(32, padding=4)

Image ကို အရင် **padding=4** pixel (သို zero) ခံပြီး 40x40 ဖြစ်သွားမယ်။ ပြီးရင် random position ကနေ 32x32 ပြန် crop ဖြတ်မယ်။

**ရည်ရွယ်ချက်:** Model က object ရဲ့ position ကို memorize မလုပ်အောင်၊ spatial invariance ရအောင် augment လုပ်တာ

In [ ]:
# --- RandomCrop ရဲ့ effect ကို ပြမယ် ---
random_crop = transforms.RandomCrop(32, padding=4)

# padding ထည့်ပြီး crop ဖြတ်တဲ့ results 5 ခု ပြမယ် (random ဖြစ်လို့ တစ်ခုနဲ့တစ်ခု ကွာမယ်)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))

# ပထမခုက original
axes[0].imshow(original_img)
axes[0].set_title("Original\n32x32", fontsize=10)
axes[0].axis('off')

# padding ခံထားတဲ့ image ကို ပြမယ် (40x40)
padded_img = transforms.Pad(4)(original_img)
axes[1].imshow(padded_img)
axes[1].set_title(f"After Padding\n{padded_img.size[0]}x{padded_img.size[1]}", fontsize=10)
axes[1].axis('off')

# RandomCrop results 4 ခု — random position ကနေ crop ဖြတ်ထားတာ
for i in range(4):
    cropped = random_crop(original_img)
    axes[i + 2].imshow(cropped)
    axes[i + 2].set_title(f"Crop #{i+1}\n{cropped.size[0]}x{cropped.size[1]}", fontsize=10)
    axes[i + 2].axis('off')

plt.suptitle("RandomCrop(32, padding=4) — ခြားနားချက်ကို သတိထားကြည့်ပါ", fontsize=12)
plt.tight_layout()
plt.show()

## Step 3: RandomHorizontalFlip()

50% probability နဲ့ image ကို ဘယ်/ညာ flip လုပ်မယ်။ ကျန်တဲ့ 50% မှာ ဘာမှ မပြောင်းဘူး။

**ရည်ရွယ်ချက်:** Model က ဘယ်ဘက်/ညာဘက် orientation ကို depend မလုပ်အောင်

In [ ]:
# --- RandomHorizontalFlip ရဲ့ effect ---
# p=1 ဆိုရင် 100% flip လုပ်မယ် (demo အတွက်)
always_flip = transforms.RandomHorizontalFlip(p=1.0)
flipped_img = always_flip(original_img)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(original_img)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(flipped_img)
axes[1].set_title("Flipped (p=1.0)")
axes[1].axis('off')

plt.suptitle("RandomHorizontalFlip — ဘယ်/ညာ ပြောင်းပြန်", fontsize=12)
plt.tight_layout()
plt.show()

## Step 4: ToTensor()

PIL Image (H, W, C) `[0, 255]` uint8 → PyTorch Tensor (C, H, W) `[0.0, 1.0]` float32

- Shape ပြောင်း: `(32, 32, 3)` → `(3, 32, 32)` — channel first format (PyTorch convention)
- Value ပြောင်း: pixel/255.0 → `[0, 1]` range

In [ ]:
# --- ToTensor() ရဲ့ effect ---
to_tensor = transforms.ToTensor()
tensor_img = to_tensor(original_img)

print("=== Before: PIL Image ===")
print(f"  Type : {type(original_img)}")
print(f"  Size : {original_img.size}")        # (W, H) format
print(f"  Range: [0, 255]")

print("\n=== After: ToTensor() ===")
print(f"  Type : {type(tensor_img)}")
print(f"  Shape: {tensor_img.shape}")          # (C, H, W) — channel first!
print(f"  Dtype: {tensor_img.dtype}")          # float32
print(f"  Range: [{tensor_img.min():.4f}, {tensor_img.max():.4f}]")  # [0, 1]

# --- Tensor ကို ပြန်ပြဖို့: (C,H,W) → (H,W,C) permute လုပ်ရမယ် ---
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].imshow(original_img)
axes[0].set_title("PIL Image\n(H,W,C) [0,255]")
axes[0].axis('off')

# Tensor ကို numpy ပြောင်းပြီး channel last format ပြန်လုပ်
axes[1].imshow(tensor_img.permute(1, 2, 0).numpy())
axes[1].set_title("ToTensor()\n(C,H,W)→permute→(H,W,C) [0,1]")
axes[1].axis('off')

# Channel တစ်ခုချင်း ပြမယ် (Red channel)
axes[2].imshow(tensor_img[0].numpy(), cmap='Reds')
axes[2].set_title("Red Channel Only\ntensor_img[0]")
axes[2].axis('off')

plt.suptitle("ToTensor() — PIL→Tensor, [0,255]→[0,1], (H,W,C)→(C,H,W)", fontsize=11)
plt.tight_layout()
plt.show()

## Step 5: Normalize(mean, std)

ToTensor() ပြီးတဲ့ [0, 1] range ကနေ → channel-wise standardization လုပ်မယ်

$$\text{normalized} = \frac{\text{pixel} - \text{mean}}{\text{std}}$$

CIFAR-10 mean/std: `(0.4914, 0.4822, 0.4465)`, `(0.2470, 0.2435, 0.2616)`

**ပြီးရင်:** pixel range က **negative values** ပါလာမယ် (≈ [-2, +2] range) — ဒါက neural network training အတွက် ပိုကောင်းတယ်

In [ ]:
# --- Normalize ရဲ့ effect ---
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)

normalize = transforms.Normalize(MEAN, STD)

# ToTensor() ပြီးသား tensor ကို normalize လုပ်မယ်
normalized_img = normalize(tensor_img)

print("=== Before: ToTensor() only ===")
print(f"  Range: [{tensor_img.min():.4f}, {tensor_img.max():.4f}]")
print(f"  Mean per channel: {tensor_img.mean(dim=[1,2]).tolist()}")

print("\n=== After: Normalize(mean, std) ===")
print(f"  Range: [{normalized_img.min():.4f}, {normalized_img.max():.4f}]")
print(f"  Mean per channel: {[f'{x:.4f}' for x in normalized_img.mean(dim=[1,2]).tolist()]}")
print(f"  Std per channel:  {[f'{x:.4f}' for x in normalized_img.std(dim=[1,2]).tolist()]}")
print(f"  ⚠️ Negative values ပါလာပြီ! → NN training အတွက် ပိုကောင်းတယ်")

# --- Visualization: Before vs After Normalize ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Before normalize (ToTensor only) — [0, 1] range, imshow က ရိုးရိုး ပြလို့ရတယ်
axes[0].imshow(tensor_img.permute(1, 2, 0).numpy())
axes[0].set_title(f"After ToTensor()\n[{tensor_img.min():.2f}, {tensor_img.max():.2f}]")
axes[0].axis('off')

# After normalize — negative values ပါလို့ imshow မှာ clamp/rescale လုပ်ရမယ်
# unnormalize ပြန်လုပ်မပြဘဲ raw normalized values ကို ပြမယ်
norm_display = normalized_img.permute(1, 2, 0).numpy()
# [min, max] → [0, 1] rescale လုပ်ပြီး ပြမယ်
norm_rescaled = (norm_display - norm_display.min()) / (norm_display.max() - norm_display.min())
axes[1].imshow(norm_rescaled)
axes[1].set_title(f"After Normalize (rescaled to display)\n[{normalized_img.min():.2f}, {normalized_img.max():.2f}]")
axes[1].axis('off')

# Unnormalize ပြန်လုပ်ရင် original ပြန်ရမယ်
mean = np.array(MEAN)
std = np.array(STD)
unnorm = normalized_img.permute(1, 2, 0).numpy() * std + mean  # ပြန်ပြောင်း
unnorm = np.clip(unnorm, 0, 1)
axes[2].imshow(unnorm)
axes[2].set_title("Unnormalized (ပြန်ပြောင်း)\nOriginal နဲ့ အတူတူ ပြန်ရမယ်")
axes[2].axis('off')

plt.suptitle("Normalize Effect — pixel values center around 0", fontsize=12)
plt.tight_layout()
plt.show()

## Step 6: Full Pipeline — Compose အကုန်လုံး တစ်ခါတည်း

`transforms.Compose` က transform တွေကို **အစဉ်လိုက်** apply လုပ်သွားတယ်:

```
PIL Image (32x32, [0,255])
  → RandomCrop(32, padding=4)    # augmentation
  → RandomHorizontalFlip()       # augmentation
  → ToTensor()                   # PIL → Tensor, [0,1]
  → Normalize(mean, std)         # standardize, ≈[-2, +2]
  → Final Tensor (3x32x32)      # model input ready!
```

In [ ]:
# --- Full Pipeline: Compose အကုန်လုံး ---
full_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 1️⃣ padding+crop
    transforms.RandomHorizontalFlip(),          # 2️⃣ 50% flip
    transforms.ToTensor(),                      # 3️⃣ PIL→Tensor [0,1]
    transforms.Normalize(MEAN, STD)             # 4️⃣ standardize
])

# helper: normalized tensor ကို display ပြန်လုပ်ဖို့
def unnormalize(img_tensor, mean=MEAN, std=STD):
    """Tensor (C,H,W) → numpy (H,W,C) [0,1] for display"""
    mean = np.array(mean)
    std = np.array(std)
    img = img_tensor.permute(1, 2, 0).numpy()
    img = img * std + mean
    return np.clip(img, 0, 1)

# --- တစ်ခု image ကို 6 ခါ transform လုပ်ပြီး ကွာခြားချက်ကို ပြမယ် ---
fig, axes = plt.subplots(1, 7, figsize=(18, 3))

# Original
axes[0].imshow(original_img)
axes[0].set_title("Original", fontsize=10, fontweight='bold')
axes[0].axis('off')

# Full pipeline 6 variations — random ဖြစ်လို့ run တိုင်း ကွာမယ်
for i in range(6):
    transformed = full_transform(original_img)  # Compose pipeline တစ်ခုလုံး run
    axes[i + 1].imshow(unnormalize(transformed))
    axes[i + 1].set_title(f"Transform #{i+1}", fontsize=10)
    axes[i + 1].axis('off')
    
    if i == 0:
        # ပထမ result ရဲ့ stats ပြမယ်
        print(f"Final tensor shape: {transformed.shape}")
        print(f"Final tensor range: [{transformed.min():.4f}, {transformed.max():.4f}]")

plt.suptitle("Same image → Full Pipeline 6 ခါ (Random augmentation ကြောင့် ကွာပါမယ်)", fontsize=12)
plt.tight_layout()
plt.show()

## Part A Summary

| Step | Transform | Input | Output | ပြောင်းလဲမှု |
|------|-----------|-------|--------|-------------|
| 1 | `RandomCrop(32, padding=4)` | PIL 32x32 | PIL 32x32 | padding ခံပြီး random position crop |
| 2 | `RandomHorizontalFlip()` | PIL 32x32 | PIL 32x32 | 50% chance flip |
| 3 | `ToTensor()` | PIL (H,W,C) [0,255] | Tensor (C,H,W) [0,1] | format + range ပြောင်း |
| 4 | `Normalize(mean, std)` | Tensor [0,1] | Tensor [≈-2, +2] | channel-wise standardize |

> **Train set** မှာပဲ RandomCrop, RandomHorizontalFlip သုံးတယ်။ **Test set** မှာ ToTensor + Normalize ပဲ သုံးတယ် (augmentation မလိုဘူး — fair evaluation ဖြစ်ဖို့)။

---

# Part B: Additional Preprocessing Techniques

အောက်မှာ ဖော်ပြမယ့် transforms တွေက **ကိုယ့် dataset/task ပေါ်** မူတည်ပြီး ရွေးသုံးရမယ့် preprocessing techniques တွေ ဖြစ်ပါတယ်။

## Step 7: Resize() & CenterCrop()

Real-world dataset တွေမှာ image size တွေ မညီတတ်ဘူး (eg. 640x480, 1920x1080, 256x256 စသည်)။ CNN က **fixed input size** လိုအပ်လို့ Resize/Crop လုပ်ရတယ်။

- `Resize(size)` — image ကို target size ပြောင်း (aspect ratio ပြောင်းနိုင်)
- `Resize(size)` + `CenterCrop(size)` — aspect ratio ထိန်းပြီး center ကနေ crop
- **Transfer Learning** မှာ pretrained model input size (224x224, 299x299) နဲ့ match ဖြစ်အောင် သုံးတယ်

In [ ]:
# --- Resize & CenterCrop ရဲ့ effect ---
# CIFAR-10 image (32x32) ကို အရွယ်စုံ ပြောင်းပြမယ်

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1: Resize (size ပြောင်း)
sizes = [16, 32, 64, 224]
for i, size in enumerate(sizes):
    resized = transforms.Resize((size, size))(original_img)
    axes[0][i].imshow(resized)
    axes[0][i].set_title(f"Resize({size}x{size})\nActual: {resized.size}", fontsize=10)
    axes[0][i].axis('off')
axes[0][0].set_ylabel("Resize", fontsize=12, fontweight='bold')

# Row 2: Resize(bigger) + CenterCrop — aspect ratio ထိန်းပြီး crop
# ပထမ Resize(40) ပြီးမှ CenterCrop(32) — အလယ်ကနေ ဖြတ်
resize_crop_sizes = [
    ("Resize(40)→CenterCrop(32)", transforms.Compose([transforms.Resize(40), transforms.CenterCrop(32)])),
    ("Resize(48)→CenterCrop(32)", transforms.Compose([transforms.Resize(48), transforms.CenterCrop(32)])),
    ("Resize(256)→CenterCrop(224)", transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])),
    ("Resize(64)→FiveCrop(32)[0]", None),  # FiveCrop demo
]

for i in range(3):
    label, t = resize_crop_sizes[i]
    result = t(original_img)
    axes[1][i].imshow(result)
    axes[1][i].set_title(f"{label}\nResult: {result.size}", fontsize=9)
    axes[1][i].axis('off')

# FiveCrop: 4 corners + center ကနေ 5 ခု crop
five_cropped = transforms.Compose([transforms.Resize(64), transforms.FiveCrop(32)])(original_img)
axes[1][3].imshow(five_cropped[0])  # top-left crop
axes[1][3].set_title(f"FiveCrop(32)\n5 crops: corners+center", fontsize=9)
axes[1][3].axis('off')

axes[1][0].set_ylabel("Resize+Crop", fontsize=12, fontweight='bold')

plt.suptitle("Resize & Crop — Image Size ပြောင်းခြင်း", fontsize=13)
plt.tight_layout()
plt.show()

print("💡 Transfer Learning သုံးရင်: Resize(256) → CenterCrop(224) — ResNet, VGG input size")
print("💡 CIFAR-10 (32x32) က ငယ်လို့ Resize မလိုအပ်ဘူး — but real datasets မှာ မဖြစ်မနေ လုပ်ရမယ်")

## Step 8: ColorJitter(brightness, contrast, saturation, hue)

Image ရဲ့ color properties (အလင်း, contrast, satီ, hue) ကို random ပြောင်းပေးတယ်။

- **brightness** — အလင်း အမှောင် တိုး/လျှော့
- **contrast** — အရောင် ခြားနားချက် တိုး/လျှော့
- **saturation** — အရောင် သိပ်သည်းမှု (vivid ↔ washed out)
- **hue** — အရောင်သန္ဓေ ပြောင်း (red→green ဆိုသလို)

**ရည်ရွယ်ချက်:** Lighting conditions ကွာခြားတဲ့ real-world images ကို handle နိုင်အောင်

In [ ]:
# --- ColorJitter ရဲ့ effect ---
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Row 1: Parameter တစ်ခုချင်း ခွဲပြ
jitter_configs = [
    ("Original", None),
    ("brightness=0.8", transforms.ColorJitter(brightness=0.8)),
    ("contrast=0.8", transforms.ColorJitter(contrast=0.8)),
    ("saturation=0.8", transforms.ColorJitter(saturation=0.8)),
    ("hue=0.3", transforms.ColorJitter(hue=0.3)),
]

for i, (title, jitter) in enumerate(jitter_configs):
    if jitter is None:
        axes[0][i].imshow(original_img)
    else:
        axes[0][i].imshow(jitter(original_img))
    axes[0][i].set_title(title, fontsize=10)
    axes[0][i].axis('off')

# Row 2: parameter အကုန်လုံး combine ပြီး random 5 ခု ပြမယ်
full_jitter = transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2)
for i in range(5):
    jittered = full_jitter(original_img)
    axes[1][i].imshow(jittered)
    axes[1][i].set_title(f"Combined #{i+1}", fontsize=10)
    axes[1][i].axis('off')

plt.suptitle("ColorJitter — Color Properties ပြောင်းလဲခြင်း (random ဖြစ်လို့ run တိုင်း ကွာမယ်)", fontsize=12)
plt.tight_layout()
plt.show()

print("💡 brightness/contrast/saturation=(0.4) → [1-0.4, 1+0.4] = [0.6, 1.4] range ထဲ random ယူ")
print("💡 hue=0.2 → [-0.2, +0.2] range ထဲ random shift")

## Step 9: RandomRotation(degrees)

Image ကို **±degrees** range ထဲ random angle နဲ့ rotate လုပ်မယ်။

- `RandomRotation(15)` → [-15°, +15°] ကြား random rotate
- `RandomRotation(90)` → [-90°, +90°] ကြား random rotate

**ရည်ရွယ်ချက်:** Object orientation ကွာခြားနိုင်တဲ့ scenarios (eg. satellite images, medical scans) မှာ robustness တိုးအောင်

> ⚠️ CIFAR-10 လို dataset မှာ rotation ကြီးကြီး သုံးရင် airplane/ship ပုံတွေ ခေါင်းမတ်မတ် ဖြစ်သွားလို့ accuracy ကျနိုင်

In [ ]:
# --- RandomRotation ရဲ့ effect ---
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Row 1: Fixed angles ပြ
fixed_angles = [0, 15, 45, 90, 180]
for i, angle in enumerate(fixed_angles):
    rotated = transforms.functional.rotate(original_img, angle)
    axes[0][i].imshow(rotated)
    axes[0][i].set_title(f"Rotate {angle}°", fontsize=10)
    axes[0][i].axis('off')

# Row 2: RandomRotation(30) → [-30°, +30°] random 5 ခု
random_rot = transforms.RandomRotation(30)
for i in range(5):
    rotated = random_rot(original_img)
    axes[1][i].imshow(rotated)
    axes[1][i].set_title(f"Random ±30° #{i+1}", fontsize=10)
    axes[1][i].axis('off')

plt.suptitle("RandomRotation — Image ကို Rotate လုပ်ခြင်း", fontsize=13)
plt.tight_layout()
plt.show()

## Step 10: RandomErasing() (Cutout)

Image ရဲ့ random rectangular region ကို **erase** (ဖျက်) လုပ်ပြီး random values နဲ့ ဖြည့်ပေးတယ်။

- Model ကို image ရဲ့ **part** ပဲ မြင်ရပေမယ့် classify တတ်အောင် train ပေးတယ်
- **Occlusion** (object ရဲ့ part ကို တခြား object ကဖုံးနေတာ) ကို handle နိုင်အောင်
- Overfitting ကို reduce လုပ်တယ် (regularization effect)

> ⚠️ `RandomErasing` က **Tensor** ပေါ်မှာ apply ရတယ် — `ToTensor()` ပြီးမှ သုံးရမယ်

In [ ]:
# --- RandomErasing ရဲ့ effect ---
# RandomErasing က Tensor ပေါ်မှာ run ရတယ် (ToTensor ပြီးမှ)
to_tensor = transforms.ToTensor()
random_erase = transforms.RandomErasing(
    p=1.0,           # probability — demo အတွက် 100%
    scale=(0.02, 0.33),  # erase area: image ရဲ့ 2%~33%
    ratio=(0.3, 3.3),    # aspect ratio of erased area
    value=0              # fill value: 0=black, 'random'=random noise
)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Row 1: value=0 (black fill) — 5 variations
tensor_img = to_tensor(original_img)
for i in range(5):
    erased = random_erase(tensor_img.clone())
    axes[0][i].imshow(erased.permute(1, 2, 0).numpy())
    axes[0][i].set_title(f"Erase (black) #{i+1}", fontsize=10)
    axes[0][i].axis('off')

# Row 2: value='random' (random noise fill) — 5 variations
random_erase_noise = transforms.RandomErasing(p=1.0, scale=(0.02, 0.33), ratio=(0.3, 3.3), value='random')
for i in range(5):
    erased = random_erase_noise(tensor_img.clone())
    axes[1][i].imshow(erased.permute(1, 2, 0).numpy())
    axes[1][i].set_title(f"Erase (random) #{i+1}", fontsize=10)
    axes[1][i].axis('off')

plt.suptitle("RandomErasing — Image ရဲ့ Part ကို Erase/Mask လုပ်ခြင်း", fontsize=13)
plt.tight_layout()
plt.show()

print("💡 Pipeline: ... → ToTensor() → Normalize() → RandomErasing() — Tensor ပေါ်မှာ apply ရတယ်")
print("💡 value=0 → black patch | value='random' → random noise patch")

## Step 11: GaussianBlur(kernel_size, sigma)

Image ကို **Gaussian blur** filter apply လုပ်ပြီး ဝါးသွားအောင် (smooth) လုပ်တယ်။

- `kernel_size` — blur filter size (odd number: 3, 5, 7...)
- `sigma` — blur strength range (ပိုကြီးရင် ပိုဝါး)

**ရည်ရွယ်ချက်:** Image noise/sharpness ကွာခြားချက်ကို handle နိုင်အောင်၊ **Self-supervised learning** (SimCLR, BYOL) မှာ အများသုံးတယ်

In [ ]:
# --- GaussianBlur ရဲ့ effect ---
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

# Original
axes[0].imshow(original_img)
axes[0].set_title("Original", fontsize=11)
axes[0].axis('off')

# kernel_size ကွာခြားချက် ပြ — ပိုကြီးရင် ပိုဝါး
blur_configs = [
    ("kernel=3, σ=(0.1,2)", transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))),
    ("kernel=5, σ=(0.1,2)", transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))),
    ("kernel=7, σ=(2,5)",   transforms.GaussianBlur(kernel_size=7, sigma=(2.0, 5.0))),
    ("kernel=11, σ=(5,10)", transforms.GaussianBlur(kernel_size=11, sigma=(5.0, 10.0))),
]

for i, (title, blur) in enumerate(blur_configs):
    blurred = blur(original_img)
    axes[i + 1].imshow(blurred)
    axes[i + 1].set_title(title, fontsize=10)
    axes[i + 1].axis('off')

plt.suptitle("GaussianBlur — kernel ကြီးလေ/sigma ကြီးလေ ပိုဝါးလေ", fontsize=13)
plt.tight_layout()
plt.show()

## Step 12: Grayscale() — Color → Grayscale

Color image (3 channels: RGB) ကို grayscale (1 channel) ပြောင်းတယ်။

- `Grayscale(num_output_channels=1)` → 1-channel grayscale
- `Grayscale(num_output_channels=3)` → 3-channel grayscale (RGB channels 3 ခုလုံး same value)
- `RandomGrayscale(p=0.2)` → 20% chance grayscale ပြောင်း (augmentation)

**သုံးရမယ့် cases:**
- Color information မလိုတဲ့ tasks (eg. edge detection, OCR)
- Model ကို color-invariant ဖြစ်အောင် train ချင်ရင် `RandomGrayscale()` သုံး

In [ ]:
# --- Grayscale ရဲ့ effect ---
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# 1. Original (RGB - 3 channels)
axes[0].imshow(original_img)
axes[0].set_title(f"Original (RGB)\nChannels: 3", fontsize=10)
axes[0].axis('off')

# 2. Grayscale 1 channel
gray_1ch = transforms.Grayscale(num_output_channels=1)(original_img)
axes[1].imshow(gray_1ch, cmap='gray')
axes[1].set_title(f"Grayscale (1ch)\nSize: {gray_1ch.size}, mode: {gray_1ch.mode}", fontsize=10)
axes[1].axis('off')

# 3. Grayscale 3 channels (pretrained models နဲ့ compatible — Conv2d(3,...) input)
gray_3ch = transforms.Grayscale(num_output_channels=3)(original_img)
axes[2].imshow(gray_3ch)
axes[2].set_title(f"Grayscale (3ch)\nRGB channels same value", fontsize=10)
axes[2].axis('off')

# 4. RandomGrayscale (augmentation) — p=0.5 ဆိုရင် 50% chance
# demo: original vs gray ကို ယှဉ်ပြ
random_gray = transforms.RandomGrayscale(p=1.0)  # 100% for demo
axes[3].imshow(random_gray(original_img))
axes[3].set_title("RandomGrayscale(p=1.0)\nAugmentation용", fontsize=10)
axes[3].axis('off')

plt.suptitle("Grayscale Conversion — Color → Gray", fontsize=13)
plt.tight_layout()
plt.show()

# Tensor shape ခြားနားချက်
print("=== Tensor Shape Comparison ===")
print(f"  RGB  → ToTensor() → {to_tensor(original_img).shape}")        # (3, 32, 32)
print(f"  Gray(1ch) → ToTensor() → {to_tensor(gray_1ch).shape}")      # (1, 32, 32)
print(f"  Gray(3ch) → ToTensor() → {to_tensor(gray_3ch).shape}")      # (3, 32, 32)
print("\n💡 Pretrained model (ResNet etc.) က Conv2d(3,...) လို့ input 3 channels လိုတယ်")
print("   → Grayscale(num_output_channels=3) သုံးရင် compatible ဖြစ်မယ်")

## Step 13: Custom Dataset Mean & Std Calculation

CIFAR-10 mean/std က hardcode ဖြစ်ပြီး well-known ဖြစ်ပေမယ့်, **ကိုယ့် custom dataset** အတွက်ဆိုရင် ကိုယ်တိုင် calculate လုပ်ရမယ်။

$$\text{mean}_c = \frac{1}{N \times H \times W} \sum_{i=1}^{N} \sum_{h,w} \text{pixel}_{i,c,h,w}$$

$$\text{std}_c = \sqrt{\frac{1}{N \times H \times W} \sum_{i=1}^{N} \sum_{h,w} (\text{pixel}_{i,c,h,w} - \text{mean}_c)^2}$$

**ဘာကြောင့် အရေးကြီးလဲ:**
- Wrong mean/std သုံးရင် → model input distribution ကွဲပြီး training slow/poor
- `.Normalize()` မသုံးဘဲ raw [0,1] ထည့်ရင် → converge ဖြစ်ချိန် ကြာတယ်

In [ ]:
# --- Custom Dataset Mean/Std Calculation ---
# CIFAR-10 ကို ToTensor() ပဲ သုံးပြီး mean/std ကိုယ်တိုင် တွက်ပြမယ်

from torch.utils.data import DataLoader

# ToTensor() only — Normalize မသုံးဘဲ [0,1] range tensor ယူ
calc_dataset = datasets.CIFAR10(root='./data', train=True, download=True,
                                 transform=transforms.ToTensor())
calc_loader = DataLoader(calc_dataset, batch_size=1000, shuffle=False, num_workers=2)

# --- Method: Batch-wise calculation ---
mean = torch.zeros(3)   # R, G, B channels
std = torch.zeros(3)
total_pixels = 0

for images, _ in calc_loader:
    # images shape: (batch_size, 3, 32, 32)
    batch_size = images.size(0)
    pixels = batch_size * images.size(2) * images.size(3)  # B * H * W

    # channel dimension (dim=1) ကို ခြွင်းပြီး mean/std တွက်
    images_flat = images.view(batch_size, 3, -1)  # (B, 3, H*W)
    mean += images_flat.mean(dim=[0, 2]) * batch_size
    std += images_flat.std(dim=[0, 2]) * batch_size
    total_pixels += batch_size

mean /= total_pixels  # dataset-wide mean
std /= total_pixels    # dataset-wide std

print("=== CIFAR-10 Mean/Std (ကိုယ်တိုင်တွက်ချက်ခြင်း) ===")
print(f"  Calculated Mean: ({mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f})")
print(f"  Calculated Std:  ({std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f})")
print()
print(f"  Known Mean:      (0.4914, 0.4822, 0.4465)")
print(f"  Known Std:       (0.2470, 0.2435, 0.2616)")
print()
print("💡 ကိုယ့် custom dataset အတွက် ဒီ code ပုံစံအတိုင်း mean/std တွက်ပြီး Normalize() မှာ ထည့်သုံးပါ")

---

## Complete Preprocessing Summary

### Core Pipeline (မဖြစ်မနေ လိုအပ်)

| Transform | Stage | ဘာလုပ်လဲ | Train | Test |
|-----------|-------|----------|:-----:|:----:|
| `Resize()` | Preprocessing | Image size ညီအောင် ပြောင်း | ✅ | ✅ |
| `ToTensor()` | Preprocessing | PIL→Tensor, [0,255]→[0,1] | ✅ | ✅ |
| `Normalize()` | Preprocessing | Channel-wise standardize | ✅ | ✅ |

### Data Augmentation (Train set only — regularization & robustness)

| Transform | ဘာလုပ်လဲ | ဘယ်အချိန်သုံးလဲ |
|-----------|----------|---------------|
| `RandomCrop()` | Random position crop | Spatial invariance လိုချင်ရင် |
| `RandomHorizontalFlip()` | ဘယ်/ညာ flip | Object orientation symmetric ဖြစ်ရင် |
| `RandomRotation()` | Random angle rotate | Rotation invariance လိုရင် |
| `ColorJitter()` | Color properties ပြောင်း | Lighting conditions ကွာနိုင်ရင် |
| `GaussianBlur()` | Image ကို blur | Noise robustness, self-supervised learning |
| `RandomGrayscale()` | Random grayscale | Color-invariant ဖြစ်ချင်ရင် |
| `RandomErasing()` | Patch erase/mask | Occlusion robustness, regularization |

### Typical Pipeline Examples

```python
# --- CIFAR-10 (small images, from scratch) ---
train_transform = Compose([
    RandomCrop(32, padding=4),
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize(mean, std)
])

# --- ImageNet / Transfer Learning (large images, pretrained) ---
train_transform = Compose([
    Resize(256),
    RandomCrop(224),
    RandomHorizontalFlip(),
    ColorJitter(0.4, 0.4, 0.4, 0.2),
    ToTensor(),
    Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    RandomErasing(p=0.25)
])

# --- Medical/Satellite (rotation matters) ---
train_transform = Compose([
    Resize(224),
    RandomRotation(90),
    RandomHorizontalFlip(),
    RandomVerticalFlip(),
    ColorJitter(brightness=0.3),
    ToTensor(),
    Normalize(custom_mean, custom_std)
])
```

> **Key Rule:** Augmentation ≠ Preprocessing
> - **Preprocessing** (Resize, ToTensor, Normalize) = Train & Test ** Both of them ** သုံးမယ်
> - **Augmentation** (RandomCrop, Flip, Jitter...) = **Train set only** — Test set ကို augment လုပ်ရင် fair evaluation မဖြစ်ဘူး